# Exploration Methodology
# For each table in the Silver layer, I follow a consistent
# four-step approach before identifying data quality issues or writing any transformation logic:

## 1) Macro Inspection
 Start by understanding the overall shape of the data — schema, columns, 
 data types, row/column counts, and a quick look at sample records.
    
## 2) Data Integrity and Uniqueness
 Check the primary key for duplicates, then review NULL values across 
 all columns to understand missing data patterns.
    
## 3) Profiling and Anomaly Detection
 Look at value distributions for key columns, 
 inspect categorical fields (like status or country), 
 and flag any unexpected or unusual values.
    
## 4) Business and Time Logic
 Check whether relationships between columns make sense from a business perspective,
 and review time consistency between related fields (e.g., created_at vs last_login).
 
---

# Under each table below, I only note specific observations from applying this methodology

In [ ]:
# SETUP

In [ ]:
if 'spark' in globals():
    spark.stop()

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("postgres_to_parquet_silver") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.driver.memory", "1g") \
    .config("spark.sql.autoBroadcastJoinThreshold", "10m") \
    .config("spark.sql.parquet.enableVectorizedReader", "true") \
    .config("spark.sql.legacy.parquet.nanosAsLong", "true") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/23 07:20:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [1]:
Pu = "file:///data/end-to-end-streaming-data-platform/bronze/postgres/ingestion_date=2026-06-05-Jun/users.parquet"
Ps = "file:///data/end-to-end-streaming-data-platform/bronze/postgres/ingestion_date2026-06-05-Jun/subscription.parquet"
Pp = "file:///data/end-to-end-streaming-data-platform/bronze/postgres/ingestion_date2026-06-05-Jun/payments.parquet"

In [4]:
dfu = spark.read.parquet(Pu).cache()
dfs = spark.read.parquet(Ps).cache()
dfp = spark.read.parquet(Pp).cache()

In [ ]:
# I'm aware of the risks of using cache, but I'm confident about the data size here

In [4]:
dfu.createOrReplaceTempView("users_table")
dfs.createOrReplaceTempView("subscriptions_table")
dfp.createOrReplaceTempView("payments_table")

In [ ]:
# EXPLORATION

In [ ]:
## users

In [ ]:
# macro inspection

In [5]:
dfu.printSchema()

root
 |-- user_id: string (nullable = true)
 |-- email: string (nullable = true)
 |-- country: string (nullable = true)
 |-- account_status: string (nullable = true)
 |-- created_at: long (nullable = true)
 |-- last_login: long (nullable = true)



In [10]:
# fix_long
#dfu= dfu.withColumn("created_at", (F.col("created_at") / 1e9).cast("timestamp")) \
        #.withColumn("last_login", (F.col("last_login") / 1e9).cast("timestamp")) 

In [6]:
dfu.count()

1000

In [6]:
len(dfu.columns)

6

In [12]:
dfu.show(5)

+--------------------+--------------------+-------+--------------+--------------------+--------------------+
|             user_id|               email|country|account_status|          created_at|          last_login|
+--------------------+--------------------+-------+--------------+--------------------+--------------------+
|55a042c8-70fb-48e...| jesse86@example.net|    MAR|       deleted|2025-08-21 13:43:...|                NULL|
|47b4d3ca-7832-439...|taylorrenee@examp...|    KSA|        active|2025-04-05 15:33:...|2025-04-29 15:33:...|
|73298054-91af-41a...|clarkchristina@ex...|    KWT|        active|2026-01-06 17:12:...|2026-01-12 17:12:...|
|792c5481-f58b-491...|emilywilson@examp...|    KSA|     suspended|2025-12-22 21:22:...|2025-12-30 21:22:...|
|2b2a2ffe-e8a5-49d...|samantha83@exampl...|    KWT|     suspended|2025-12-08 22:15:...|2025-12-10 22:15:...|
+--------------------+--------------------+-------+--------------+--------------------+--------------------+
only showing top 5 

In [31]:
dfu.show(5,truncate=False,vertical=True)

-RECORD 0----------------------------------------------
 user_id        | 55a042c8-70fb-48e7-bbab-fca7a7628dde 
 email          | jesse86@example.net                  
 country        | MAR                                  
 account_status | deleted                              
 created_at     | 1755783833681172000                  
 last_login     | NULL                                 
-RECORD 1----------------------------------------------
 user_id        | 47b4d3ca-7832-439e-b62d-aab83ef37521 
 email          | taylorrenee@example.org              
 country        | KSA                                  
 account_status | active                               
 created_at     | 1743867197681172000                  
 last_login     | 1745940797681172000                  
-RECORD 2----------------------------------------------
 user_id        | 73298054-91af-41ac-a7bc-2dea72c8c42e 
 email          | clarkchristina@example.com           
 country        | KWT                           

In [ ]:
## integrity & uniqueness

In [7]:
# primary key duplication
dfu.groupBy("user_id").count().filter("count > 1").show()

+-------+-----+
|user_id|count|
+-------+-----+
+-------+-----+



In [13]:
# null
dfu.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in dfu.columns
]).show()

+-------+-----+-------+--------------+----------+----------+
|user_id|email|country|account_status|created_at|last_login|
+-------+-----+-------+--------------+----------+----------+
|      0|   33|      0|             0|         0|       257|
+-------+-----+-------+--------------+----------+----------+



In [ ]:
## profiling & outliers

In [43]:
dfu.groupBy("account_status").count().show()

+--------------+-----+
|account_status|count|
+--------------+-----+
|       deleted|  127|
|        active|  630|
|     suspended|  243|
+--------------+-----+



In [44]:
dfu.groupBy("country").count().show()

+-------+-----+
|country|count|
+-------+-----+
|    EGY|  141|
|    JOR|  115|
|unknown|   25|
|    QAT|  122|
|    KSA|  236|
|    UAE|  146|
|    MAR|   97|
|    KWT|  118|
+-------+-----+



In [ ]:
## business & time logic

In [49]:
dfu.filter(
    (F.col("last_login") < F.col("created_at"))
).count()

27

In [50]:
dfu.filter(
    F.col("email").isNull()&
    F.col("last_login").isNotNull()
).count()

19

In [51]:
dfu.groupBy("account_status").agg(
    F.count("email").alias("email_count"),
    F.count("last_login").alias("login_count")
).show()

+--------------+-----------+-----------+
|account_status|email_count|login_count|
+--------------+-----------+-----------+
|       deleted|        118|          0|
|        active|        618|        630|
|     suspended|        231|        113|
+--------------+-----------+-----------+



In [54]:
dfu.filter(
    (F.col("account_status") == "active") & (F.col("email").isNull())).count()

12

In [56]:
dfu.filter(
   (F.col("last_login") < F.col("created_at"))
).count()

27

In [ ]:
# Users Table

# id is a long, unwieldy identifier (UUID/long string format).
# Some timestamp columns are in nanoseconds, making them hard to read directly.
# NULL values found in email and last_login columns.
# country column contains "Unknown" values.
# Some last_login values are earlier than created_at.

In [ ]:
## subscriptions

In [ ]:
#* macro inspection

In [13]:

dfs.printSchema()

root
 |-- subscription_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- plan_type: string (nullable = true)
 |-- start_date: long (nullable = true)
 |-- end_date: long (nullable = true)
 |-- is_active: boolean (nullable = true)
 |-- auto_renew: boolean (nullable = true)



In [58]:
dfs.show(5,truncate=False,vertical=True)

-RECORD 0-----------------------------------------------
 subscription_id | 1e51e8b6-223d-4ac3-aa20-ccd3da41d90d 
 user_id         | 55a042c8-70fb-48e7-bbab-fca7a7628dde 
 plan_type       | premium                              
 start_date      | 1760976105000000000                  
 end_date        | 1768752105000000000                  
 is_active       | false                                
 auto_renew      | true                                 
-RECORD 1-----------------------------------------------
 subscription_id | 82cc6ec2-e6f3-4be1-9dc8-5ef3ea5160c2 
 user_id         | 47b4d3ca-7832-439e-b62d-aab83ef37521 
 plan_type       | free                                 
 start_date      | 1770028042000000000                  
 end_date        | NULL                                 
 is_active       | true                                 
 auto_renew      | false                                
-RECORD 2-----------------------------------------------
 subscription_id | db85095a-8c6

In [59]:
dfs.count()

1000

In [12]:
len(dfs.columns)

7

In [ ]:
#* integrity & uniqueness

In [14]:
dfs.groupBy(["user_id","subscription_id"]).count().filter("count > 1").show()

+-------+---------------+-----+
|user_id|subscription_id|count|
+-------+---------------+-----+
+-------+---------------+-----+



In [16]:
# null
dfs.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in dfs.columns
]).show()

+---------------+-------+---------+----------+--------+---------+----------+
|subscription_id|user_id|plan_type|start_date|end_date|is_active|auto_renew|
+---------------+-------+---------+----------+--------+---------+----------+
|              0|      0|        0|         0|     511|        0|         0|
+---------------+-------+---------+----------+--------+---------+----------+



In [ ]:
## profiling & outliers

In [63]:
dfs.groupBy("plan_type").count().show()

+---------+-----+
|plan_type|count|
+---------+-----+
|     free|  511|
|  premium|  489|
+---------+-----+



In [64]:
dfs.groupBy("is_active").count().show()

+---------+-----+
|is_active|count|
+---------+-----+
|    false|  293|
|     true|  707|
+---------+-----+



In [65]:
dfs.groupBy("auto_renew").count().show()

+----------+-----+
|auto_renew|count|
+----------+-----+
|      true|  244|
|     false|  756|
+----------+-----+



In [ ]:
## business & time logic

In [17]:
dfs.filter(
    (F.col("end_date") < F.col("start_date"))
).show(5,truncate=False,vertical=True)

-RECORD 0-----------------------------------------------
 subscription_id | b300393b-2538-483a-b922-03d75c09ebb8 
 user_id         | ffac40a0-19b1-4966-8fe8-ab95bd0ea58e 
 plan_type       | premium                              
 start_date      | 1755927419000000000                  
 end_date        | 1755581819000000000                  
 is_active       | false                                
 auto_renew      | true                                 
-RECORD 1-----------------------------------------------
 subscription_id | c5981390-8052-45ca-90b4-7d09c0cf7644 
 user_id         | b65c3349-1eaa-47d4-83c5-0791e4cddeb8 
 plan_type       | premium                              
 start_date      | 1762472975000000000                  
 end_date        | 1762213775000000000                  
 is_active       | false                                
 auto_renew      | false                                
-RECORD 2-----------------------------------------------
 subscription_id | 1cda17ae-d64

In [72]:
dfs.filter(
    F.col("end_date").isNotNull()&
    (F.col("end_date") < F.col("start_date"))
).show(5,truncate=False,vertical=True)

-RECORD 0-----------------------------------------------
 subscription_id | b300393b-2538-483a-b922-03d75c09ebb8 
 user_id         | ffac40a0-19b1-4966-8fe8-ab95bd0ea58e 
 plan_type       | premium                              
 start_date      | 1755927419000000000                  
 end_date        | 1755581819000000000                  
 is_active       | false                                
 auto_renew      | true                                 
-RECORD 1-----------------------------------------------
 subscription_id | c5981390-8052-45ca-90b4-7d09c0cf7644 
 user_id         | b65c3349-1eaa-47d4-83c5-0791e4cddeb8 
 plan_type       | premium                              
 start_date      | 1762472975000000000                  
 end_date        | 1762213775000000000                  
 is_active       | false                                
 auto_renew      | false                                
-RECORD 2-----------------------------------------------
 subscription_id | 1cda17ae-d64

In [18]:
dfs.filter(
    (F.col("end_date") < F.col("start_date"))
).count()

29

In [ ]:
# Subscriptions Table

# Same id format issue (long, unwieldy identifier).
# Same nanosecond timestamp issue.
# High number of NULL values in end_date.
# 29 cases where end_date is earlier than start_date.

In [ ]:
## payments

In [19]:
#* macro inspection

In [20]:
dfp.printSchema()

root
 |-- payment_id: string (nullable = true)
 |-- subscription_id: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- payment_date: long (nullable = true)
 |-- payment_status: string (nullable = true)



In [21]:
dfp.count()

489

In [22]:
len(dfp.columns)

6

In [26]:
dfp.show(5,truncate=False,vertical=True)

-RECORD 0-----------------------------------------------
 payment_id      | 2e2d403f-6abc-4022-8034-6b4fb15ffae3 
 subscription_id | 1e51e8b6-223d-4ac3-aa20-ccd3da41d90d 
 amount          | 29.99                                
 currency        | USD                                  
 payment_date    | 1760976105000000000                  
 payment_status  | refunded                             
-RECORD 1-----------------------------------------------
 payment_id      | df38bca8-b67a-427c-976f-79b8a3bd4346 
 subscription_id | db85095a-8c69-4864-90e1-99b2ba242edf 
 amount          | 29.99                                
 currency        | USD                                  
 payment_date    | 1771069316000000000                  
 payment_status  | success                              
-RECORD 2-----------------------------------------------
 payment_id      | fdbb5a49-5d39-423a-9f92-a0d9b78f0f33 
 subscription_id | 31ed3f37-7b03-4d4c-80fa-74f4bb85b997 
 amount          | 14.99       

In [ ]:
#* integrity & uniqueness

In [24]:
dfp.groupBy(["payment_id","subscription_id"]).count().filter("count > 1").show()

+----------+---------------+-----+
|payment_id|subscription_id|count|
+----------+---------------+-----+
+----------+---------------+-----+



In [25]:
dfp.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in dfp.columns
]).show()

+----------+---------------+------+--------+------------+--------------+
|payment_id|subscription_id|amount|currency|payment_date|payment_status|
+----------+---------------+------+--------+------------+--------------+
|         0|              0|     0|       0|           0|             0|
+----------+---------------+------+--------+------------+--------------+



In [ ]:
## profiling & outliers

In [27]:
dfp.groupBy("amount").count().show()

+------+-----+
|amount|count|
+------+-----+
| 14.99|  157|
|  9.99|  165|
|-14.99|    5|
| -9.99|    9|
| 29.99|  151|
|-29.99|    2|
+------+-----+



In [28]:
dfp.groupBy("currency").count().show()

+--------+-----+
|currency|count|
+--------+-----+
|     USD|  489|
+--------+-----+



In [29]:
dfp.groupBy("payment_status").count().show()

+--------------+-----+
|payment_status|count|
+--------------+-----+
|      refunded|   23|
|       success|  417|
|        failed|   49|
+--------------+-----+



In [ ]:
## business & time logic

In [35]:
dfp.filter(
    (F.col("payment_status") == "success") & (F.col("amount") < 0)
).count()

14

In [36]:
dfp.filter(
    (F.col("amount") < 0)
).count()

16

In [ ]:
# Payments Table

# Same id format issue (long, unwieldy identifier).
# Same nanosecond timestamp issue.
# No NULL values found.
# 16 cases of negative payment amounts, 14 of which are in "success" stat